# 02 · Load landing: repo files → landing Volume
Copies the source CSV files from `data/source/` in this repo into the landing Volume, laid out like a client delivery:
`landing/<system>/<dataset>/delivery_date=<date>/<file>.csv`

**Why not upload by hand?** Because this notebook is code: re-runnable, versioned, and identical in any workspace (Free Edition, sandbox, client).
**Re-runnable:** running it again overwrites the same files and creates no duplicates.

In [0]:
%python
# Widgets = notebook parameters. They appear as input boxes at the top of the notebook,
# and a Job can pass different values later without changing any code.
dbutils.widgets.text("catalog", "retail_demo")
dbutils.widgets.text("schema", "fashion_retail_inventory")
dbutils.widgets.text("delivery_date", "2026-11-24")   # date of the source export

catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
delivery_date = dbutils.widgets.get("delivery_date")

In [0]:
%python
import os

# In Databricks, the working directory of a notebook is the folder it lives in,
# which here is the repo root. So this works in any workspace and for any user.
repo_root   = os.getcwd()
source_dir  = os.path.join(repo_root, "data", "source")
landing_dir = f"/Volumes/{catalog}/{schema}/landing"

assert os.path.isdir(source_dir), f"Not found: {source_dir}. Is this notebook at the repo root, and did you Pull?"

datasets = []
for root, _, files in os.walk(source_dir):
    for f in sorted(files):
        if f.endswith(".csv"):
            datasets.append((os.path.relpath(root, source_dir), f))   # e.g. ('pos/sales_lines', 'sm_sales_lines.csv')

print(f"Found {len(datasets)} files in {source_dir}")
for rel, f in sorted(datasets):
    print(f"  {rel:<28} {f}")

In [0]:
%python
import os

# In Databricks, the working directory of a notebook is the folder it lives in,
# which here is the repo root. So this works in any workspace and for any user.
repo_root   = os.getcwd()
source_dir  = os.path.join(repo_root, "data", "source")
landing_dir = f"/Volumes/{catalog}/{schema}/landing"

assert os.path.isdir(source_dir), f"Not found: {source_dir}. Is this notebook at the repo root, and did you Pull?"

datasets = []
for root, _, files in os.walk(source_dir):
    for f in sorted(files):
        if f.endswith(".csv"):
            datasets.append((os.path.relpath(root, source_dir), f))   # e.g. ('pos/sales_lines', 'sm_sales_lines.csv')

print(f"Found {len(datasets)} files in {source_dir}")
for rel, f in sorted(datasets):
    print(f"  {rel:<28} {f}")

In [0]:
%python
import shutil

# Volumes behave like a normal file system on Databricks compute,
# so plain Python file operations work on /Volumes/... paths.
for rel, f in datasets:
    target_dir = os.path.join(landing_dir, rel, f"delivery_date={delivery_date}")
    os.makedirs(target_dir, exist_ok=True)
    shutil.copyfile(os.path.join(source_dir, rel, f), os.path.join(target_dir, f))

print(f"Copied {len(datasets)} files to {landing_dir}")

In [0]:
%python
rows = []
for root, _, files in os.walk(landing_dir):
    for f in files:
        p = os.path.join(root, f)
        rows.append((os.path.relpath(p, landing_dir), round(os.path.getsize(p) / 1024, 1)))

display(spark.createDataFrame(sorted(rows), "path STRING, size_kb DOUBLE"))

In [0]:
%sql
-- read_files reads files directly with SQL, without creating a table.
-- _metadata is a hidden column with details about the file each row came from.
SELECT delivery_date,
       _metadata.file_name,
       COUNT(*)              AS row_count,
       MIN(bill_date)        AS first_bill,
       MAX(bill_date)        AS last_bill
FROM read_files('/Volumes/retail_demo/fashion_retail_inventory/landing/pos/sales_lines/',
                format => 'csv', header => true)
GROUP BY ALL;